In [38]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime, timedelta
from tqdm import tqdm
import re
import locale
import urllib3

# Suppress the InsecureRequestWarning specifically
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [39]:
def get_html(url = 'https://sindipetro-es.org.br/noticias-2/'):
    payload = {}
    headers = {}

    response = requests.request("GET", url, headers=headers, data=payload, verify=False)
    html_content = response.text

    return html_content

In [40]:
def get_links_and_dates(html_content):
    soup = BeautifulSoup(html_content, 'html.parser')
    spans = soup.find_all('span', class_='post_meta_item post_date')

    locale.setlocale(locale.LC_TIME, "pt_BR.utf8") 

    news_links = []

    for span in spans:
        a = span.find('a')
        link = a.get('href')
        date = span.text.strip()
        date = datetime.strptime(date, "%d/%m/%Y")
            
        link_date = [link, date]
        news_links.append(link_date)

    return news_links


In [41]:
def get_validated_links(news_links, min_date = datetime(2025,6,1)):
    validated_links = []
    for link, date in news_links:
        if date < min_date:
            break
        else:
            validated_links.append([link, date])

    return validated_links

In [42]:
def get_content_news(url):
    html_content = get_html(url)
    soup = BeautifulSoup(html_content, 'html.parser')

    title = soup.find('h1').text
    div = soup.find('div', class_='post_content post_content_single entry-content')

    paragraphs = div.find_all('p')
    return title, paragraphs

In [43]:
def main():
    url = 'https://sindipetro-es.org.br/noticias-2/'
    html_content = get_html(url)
    news_links = get_links_and_dates(html_content)
    validated_news_links = get_validated_links(news_links)

    result = []
    for url, date in tqdm(validated_news_links):
        title, paragraphs = get_content_news(url)
        num_paragraph = 1
        for paragraph in paragraphs:
            result.append(
                {
                    'sindicato': 'ES',
                    'url' : url,
                    'titulo' : title,
                    'data': date,
                    'paragrafo' : paragraph.text,
                    'num_paragrafo' : num_paragraph
                }
            )
            num_paragraph += 1

    return result

In [44]:
result = main()

df = pd.DataFrame(result)
df
#df2 = df.explode('paragrafo')
#df2.to_dict('records')

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 11/11 [00:19<00:00,  1.74s/it]


,sindicato,url,titulo,data,paragrafo,num_paragrafo
0,ES,https://sindipetro-es.org.br/2025/08/12/sindip...,Sindipetro-ES se reúne com RH local da UN-ES p...,2025-08-12,O Sindipetro-ES esteve em atividade online com...,1
1,ES,https://sindipetro-es.org.br/2025/08/12/sindip...,Sindipetro-ES se reúne com RH local da UN-ES p...,2025-08-12,,2
2,ES,https://sindipetro-es.org.br/2025/08/12/sindip...,Sindipetro-ES se reúne com RH local da UN-ES p...,2025-08-12,📍Posto Avançado APS no Edivit: Foi solicitada ...,3
3,ES,https://sindipetro-es.org.br/2025/08/12/sindip...,Sindipetro-ES se reúne com RH local da UN-ES p...,2025-08-12,📍Acompanhamento da frequência do COI: Atualiz...,4
4,ES,https://sindipetro-es.org.br/2025/08/12/sindip...,Sindipetro-ES se reúne com RH local da UN-ES p...,2025-08-12,"📍Internet UMS: Na era da informação, é inadmis...",5
...,...,...,...,...,...,...
70,ES,https://sindipetro-es.org.br/2025/06/13/trabal...,Trabalhadores da Texcal suspendem greve e nego...,2025-06-13,"Os trabalhadores da Texcal, empresa prestadora...",1
71,ES,https://sindipetro-es.org.br/2025/06/13/trabal...,Trabalhadores da Texcal suspendem greve e nego...,2025-06-13,Após diversas tentativas de negociação direta ...,2
72,ES,https://sindipetro-es.org.br/2025/06/13/trabal...,Trabalhadores da Texcal suspendem greve e nego...,2025-06-13,"A mobilização demonstrou, mais uma vez, a uniã...",3
73,ES,https://sindipetro-es.org.br/2025/06/13/trabal...,Trabalhadores da Texcal suspendem greve e nego...,2025-06-13,"Seguimos na luta por respeito, valorização e d...",4


In [53]:
linha = df['paragrafo'][2]

In [60]:
print(r'{}'.format(linha[0]))

📍
